## **Imports**
---

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch

DEVICE: torch.device = torch.accelerator.current_accelerator(check_available=True)
if DEVICE is None:
    DEVICE = torch.device("cpu")

print(DEVICE)

In [ ]:
import tactex
from tactex.reconstruction.unet import UNet
from tactex.sensing import raycasting
from tactex.settings import get_settings
from tactex.typing import OccupancyGrid2D
from tactex.visualization import prediction as prediction_vis
from tactex.visualization import progression as progression_vis

print(tactex.__file__)

In [ ]:
import functools
import statistics
import time
from collections.abc import Callable
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import MetricCollection, MetricTracker

## **Model Loading**
---

In [ ]:
# model_path: Path = Path(r"G:\My Drive\tum-adlr-13\models\overfit\epoch_000500\unet.pt") # 03797390
# model_path: Path = Path(r"G:\My Drive\tum-adlr-13\models\training\epoch_000250\unet.pt") # 03797390
model_path: Path = Path(r"G:\My Drive\tum-adlr-13\models\training\epoch_000021\unet.pt")  # circles
 
model_path

In [ ]:
obj = torch.load(model_path, map_location=DEVICE)
obj.keys()

In [ ]:
model: UNet = UNet(in_channels=3, out_channels=1, widths=obj["widths"], kernel_size=obj["kernel_size"])
result = model.load_state_dict(obj["model_state_dict"])
result

In [ ]:
# occupancy_grids = np.load(r"C:\dev\tum-adlr-13\data\datasets\03797390\res064\v001\grids.npy")
occupancy_grids = np.load(r"C:\dev\tum-adlr-13\data\datasets\circles\res064\v001\grids.npy")
# occupancy_grids = np.load(r"C:\dev\tum-adlr-13\data\generated\shapenet\03797390\res064\v001\occupancy_grid.npz")["masks"]
# occupancy_grids = np.load(r"C:\dev\tum-adlr-13\data\generated\rectangle\res064\v001\occupancy.npy")


# occupancy_grids = occupancy_grids[:2, ...] # 03797390 - overfit
# occupancy_grids = occupancy_grids[np.random.default_rng().integers(low=0, high=len(occupancy_grids), size=(64)), ...]
# occupancy_grids = occupancy_grids[[57, 11, 41, 15, 59, 66, 51, 56, 31, 30], ...] # 03797390 - 250
occupancy_grids = occupancy_grids[[4298, 624, 1383, 3785, 1950, 1377, 2953, 263, 1265, 3824], ...] # circles - 500
B, H, W = occupancy_grids.shape
print("Occupancy Grids Shape", occupancy_grids.shape)

In [ ]:
rays = raycasting.sample(resolution=64, b=B, n=50, rng=np.random.default_rng())
print("Rays Shape", rays.shape)

In [ ]:
traversals = raycasting.traverse(rays=rays, resolution=64)
_, N, M, _ = traversals.shape
print("Traversals Shape", traversals.shape)

In [ ]:
num_rays: np.ndarray = np.random.default_rng().integers(low=N - 1, high=N, size=(B, 1))
active_rays: np.ndarray = np.arange(N) < num_rays  # (B, N)
traversal_grids = raycasting.cast(occupancy_grids=occupancy_grids, traversals=traversals, num_rays=num_rays)
print("Traversal Grids Shape", traversal_grids.shape)

## **Visualization**
---

In [ ]:
prediction_vis.show(model=model, traversal_grids=traversal_grids, occupancy_grids=occupancy_grids, device=DEVICE)

In [ ]:
fig = progression_vis.show(model=model, occupancy_grids=occupancy_grids, snapshots=50, snapshot_interval=5, probabilities=True, device=DEVICE)